# Lab — Recommandations musicales locales

**Solution : MusicCaps → index TF-IDF local → recherche des extraits → génération ancrée via Ollama.**
Le catalogue contient de vraies descriptions musicales ; aucun fichier audio n’est téléchargé. Les extraits sont identifiés par leur ID YouTube et leur intervalle, car MusicCaps ne fournit pas de titres ou d’artistes fiables. Les liens peuvent devenir indisponibles.

La recherche fonctionne sans clé API. Le mode `RUN_LOCAL_LLM=True` active un vrai RAG avec un modèle Ollama local. Sinon, le notebook affiche les éléments retrouvés : cette restitution de sources n’est pas présentée comme une génération LLM.

La démonstration ESC-50/PANNs/Pinecone du support est conservée en annexe facultative ; elle porte sur des sons environnementaux et ne constitue pas le moteur musical demandé.


In [1]:
# Installer une fois : %pip install -r requirements.txt
RUN_LOCAL_LLM = False
RUN_AUDIO_DEMO = False
OLLAMA_MODEL = 'qwen2.5:3b'
from music_rag import MusicRetriever, load_catalog, render_evidence, generate_local
catalog = load_catalog()
retriever = MusicRetriever(catalog)
print(f'{len(catalog)} extraits MusicCaps indexés localement.')
print('Matrice TF-IDF :', retriever.matrix.shape)


5521 extraits MusicCaps indexés localement.
Matrice TF-IDF : (5521, 84658)


## Recherche selon les préférences
La similarité cosinus compare des vecteurs TF-IDF de descriptions et de tags, pas des embeddings audio. Une petite expansion transparente du vocabulaire français est fournie ; l’anglais reste mieux couvert. Les filtres explicites s’appliquent aux métadonnées, sans certifier le contenu sonore.


In [2]:
preferences = 'guitare acoustique douce et relaxante'
hits = retriever.search(preferences,k=3,required_terms=['guitar'])
print(render_evidence(hits))


[Vwekk3EOa-U:40] low quality, smooth bass, electric guitar melody, synth pad chords, mellow piano chords, soft
Description source : The low quality recording features a smooth bass, electric guitar melody, synth pad chords and mellow piano chords playing. It sounds soft, mellow and easygoing.
Écouter : https://www.youtube.com/watch?v=Vwekk3EOa-U&t=40
Score lexical : 0.244

[7FHzw4HV75Y:230] regional mexican, soft piano melody, acoustic rhythm guitar, smooth bass, passionate male vocal, addictive brass melody
Description source : The Regional Mexican song features a soft piano melody, rhythm acoustic guitar, smooth bass, addictive brass melody and passionate male vocal singing on top of it. It sounds soft, mellow, passionate and emotional.
Écouter : https://www.youtube.com/watch?v=7FHzw4HV75Y&t=230
Score lexical : 0.233

[_yRFCl-z-EE:190] low quality, harmonizing wide female vocals, acoustic rhythm guitar, traditional, soft, mellow
Description source : The low quality recording features

## Trois usages


In [3]:
for preference in ['calm relaxing piano instrumental', 'energetic electronic dance drums', 'jazz saxophone double bass']:
    print('\nPRÉFÉRENCE :',preference)
    for hit in retriever.search(preference,k=3):
        print(hit['id'],hit['score'],', '.join(hit['aspects'][:5]),hit['url'])



PRÉFÉRENCE : calm relaxing piano instrumental
fTGZEmn3BY4:30 0.172603 hindustani classical music, violin, tanpura, calm, relaxing https://www.youtube.com/watch?v=fTGZEmn3BY4&t=30
R5KBk76b9HE:110 0.150379 piano cover, glam metal, keyboard, piano sound, gentle playing https://www.youtube.com/watch?v=R5KBk76b9HE&t=110
SvsCM0fLM5g:30 0.129905 country music, no singer, instrumental, steel guitar, mellow tune https://www.youtube.com/watch?v=SvsCM0fLM5g&t=30

PRÉFÉRENCE : energetic electronic dance drums
r1W1z_31Obw:120 0.274838 electronic dance music, house, female vocals, melodic singing, keyboard https://www.youtube.com/watch?v=r1W1z_31Obw&t=120
NuN-ug3dIkw:30 0.206077 rave, electronic dance music, electronic drums, synth, dance https://www.youtube.com/watch?v=NuN-ug3dIkw&t=30
G22YfD5xxMU:60 0.200827 electronic dance, live performance, amateur recording, vocal sample, synth https://www.youtube.com/watch?v=G22YfD5xxMU&t=60

PRÉFÉRENCE : jazz saxophone double bass
8sSV_vqOlS4:250 0.368626 l

## Recommandations similaires sans retourner l’extrait de départ


In [4]:
seed = hits[0]['id']
similar = retriever.similar(seed,k=3)
assert all(row['id'] != seed for row in similar)
print('Extrait de départ :', seed)
print(render_evidence(similar))


Extrait de départ : Vwekk3EOa-U:40
[r4G71I1dFpA:160] low quality, soft echoing female vocal, mellow piano chords, mellow synth pad chords, reverberant percussion, soft
Description source : The low quality recording features a soft echoing female vocal singing over mellow piano chords, mellow synth pad chords and reverberant percussion. It sounds soft, mellow, passionate, emotional and sad.
Écouter : https://www.youtube.com/watch?v=r4G71I1dFpA&t=160
Score lexical : 0.395

[Nlg8AbWRV_c:30] low quality, synth pad chords, mellow arpeggiated piano melody, soft, emotional
Description source : The low quality recording features synth pad chords, followed by mellow arpeggiated piano melody. It sounds soft and emotional.
Écouter : https://www.youtube.com/watch?v=Nlg8AbWRV_c&t=30
Score lexical : 0.352

[PQXYWc3JHhU:30] low quality, noisy, arpeggiated synth melody, synth pad chords, mellow, soft
Description source : The low quality recording features an arpeggiated synth melody, followed by synth

## Génération locale avec contexte retrouvé
Installer Ollama, puis exécuter `ollama pull qwen2.5:3b` et démarrer son service local. Passer ensuite `RUN_LOCAL_LLM=True`.
Le générateur ne peut sélectionner que les IDs retrouvés et doit fournir une citation exacte de leur description. La validation rejette les IDs inventés, doublons et justifications sans source. Elle garantit l’ancrage des champs, pas la pertinence subjective de l’écoute.


In [5]:
if RUN_LOCAL_LLM:
    import json
    print(json.dumps(generate_local(preferences,hits,model=OLLAMA_MODEL),ensure_ascii=False,indent=2))
else:
    print('Ollama non exécuté : les résultats précédents proviennent de la recherche locale réelle.')


Ollama non exécuté : les résultats précédents proviennent de la recherche locale réelle.


## Vérifications


In [6]:
assert len(catalog) == 5521
assert len({row['id'] for row in catalog}) == len(catalog)
assert retriever.search('zzxqvunknownword') == []
assert retriever.search('piano',required_terms=['zzxqvunknownword']) == []
assert len(hits) == 3
assert all(hit['score'] > 0 for hit in hits)
assert all('guitar' in (hit['caption']+' '.join(hit['aspects'])).lower() for hit in hits)
print('Catalogue, scores, filtres, absence de résultats et exclusion : validés.')


Catalogue, scores, filtres, absence de résultats et exclusion : validés.


## Bilan
Le moteur recherche réellement dans 5 521 extraits MusicCaps et produit des recommandations sourcées. Les filtres et l’exclusion de l’extrait de départ sont testés ; la génération Ollama est implémentée mais non exécutée dans cet environnement. La recherche lexicale peut manquer les synonymes, mal interpréter une négation ou favoriser une description longue ; les scores ne sont pas des probabilités de satisfaction.

Pour améliorer le système : comparer TF-IDF à des embeddings multilingues, évaluer la pertinence avec des préférences annotées et intégrer des embeddings audio si les fichiers sont disponibles. Aucun taux de qualité ou résultat d’écoute n’est inventé.

**Sources :** [MusicCaps, Google](https://huggingface.co/datasets/google/MusicCaps), [API Ollama](https://docs.ollama.com/api/generate). Les métadonnées MusicCaps sont distribuées sous CC BY-SA 4.0 ; voir `data/ATTRIBUTION.md`.


# Annexe — démonstration audio du support
Pour l’exécuter, installer `requirements-audio.txt`, configurer `PINECONE_API_KEY`, puis activer `RUN_AUDIO_DEMO`. Cette partie télécharge ESC-50 et les poids PANNs et crée un index Pinecone. Elle n’est pas nécessaire à la solution locale ci-dessus et n’a pas été testée ici. Les audios sont rééchantillonnés à 32 kHz et le vecteur de requête Pinecone est aplati.


This lab demonstrate how to use Pinecone as the vector DB within an audio search application. Audio search can be used to find songs and metadata within a catalog, finding similar sounds in an audio library, or detecting who's speaking in an audio file.

We will index a set of audio recordings as vector embeddings. These vector embeddings are rich, mathematical representations of the audio recordings, making it possible to determine how similar the recordings are to one another. We will then take some new (unseen) audio recording, search through the index to find the most similar matches, and play the returned audio in this notebook.

# Install Dependencies

In [7]:
if RUN_AUDIO_DEMO:
    pass
    #!pip install librosa
    #!pip install panns-inference


In [8]:
if RUN_AUDIO_DEMO:
    pass
    # !pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa


# Load Dataset

In this demo, we will use audio from the *ESC-50 dataset* — a labeled collection of 2000 environmental audio recordings, which are 5-second-long each. The dataset can be loaded from the HuggingFace model hub as follows:

In [9]:
if RUN_AUDIO_DEMO:
    pass
    from datasets import load_dataset
    from datasets import Audio as DatasetAudio
    data = load_dataset('ashraq/esc50', split='train').cast_column('audio', DatasetAudio(sampling_rate=32000))
    data


Les extraits sont rééchantillonnés en mono à 32 kHz pour le modèle PANNs.


In [10]:
if RUN_AUDIO_DEMO:
    pass
    # select the audio feature and display top three
    audios = data["audio"]
    audios[:3]


We only need the Numpy arrays as these contain all of the audio data. We will later input these Numpy arrays directly into our embedding model to generate audio embeddings.

In [11]:
if RUN_AUDIO_DEMO:
    pass
    import numpy as np
    audios = np.stack([a['array'].astype(np.float32) for a in data['audio']])


# Load Audio Embedding Model

We will use an audio tagging model trained from *PANNs: Large-Scale Pretrained Audio Neural Networks for Audio Pattern Recognition* paper to generate our audio embeddings. We use the *panns_inference* Python package, which provides an easy interface to load and use the model.

In [12]:
if RUN_AUDIO_DEMO:
    pass
    import torch
    from panns_inference import AudioTagging
    model = AudioTagging(checkpoint_path=None, device='cuda' if torch.cuda.is_available() else 'cpu')


## Initializing the Index

Now we need a place to store these embeddings and enable a efficient vector search through them all. To do that we use Pinecone, we can get a [free API key](https://app.pinecone.io/) and enter it below where we will initialize our connection to Pinecone and create a new index.

In [13]:
if RUN_AUDIO_DEMO:
    pass
    from dotenv import load_dotenv
    import os
    load_dotenv()
    PINECONE_API_KEY = os.environ['PINECONE_API_KEY']


In [14]:
if RUN_AUDIO_DEMO:
    pass
    import os
    from pinecone import Pinecone

    # configure client
    pc = Pinecone(api_key=PINECONE_API_KEY)


Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [15]:
if RUN_AUDIO_DEMO:
    pass
    from pinecone import ServerlessSpec

    cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
    region = os.environ.get('PINECONE_REGION') or 'us-east-1'

    spec = ServerlessSpec(cloud=cloud, region=region)


Create the index:

In [16]:
if RUN_AUDIO_DEMO:
    pass
    index_name = 'ironhack-audio-search-demo'


In [17]:
if RUN_AUDIO_DEMO:
    pass
    import time
    if index_name not in pc.list_indexes().names():
        pc.create_index(name=index_name,dimension=2048,metric='cosine',spec=spec)
    deadline = time.monotonic() + 120
    while not pc.describe_index(index_name).status['ready']:
        if time.monotonic() > deadline:
            raise TimeoutError('Pinecone index not ready after 120 seconds')
        time.sleep(1)
    index = pc.Index(index_name)
    print(index.describe_index_stats())


# Generate Embeddings and Upsert

Now we generate the embeddings using the audio embedding model. We must do this in batches as processing all items at once will exhaust machine memory limits and API request limits.

In [18]:
if RUN_AUDIO_DEMO:
    pass
    from tqdm.auto import tqdm

    # we will use batches of 64
    batch_size = 8

    for i in tqdm(range(0, len(audios), batch_size)):
        # find end of batch
        i_end = min(i+batch_size, len(audios))
        # extract batch
        batch = audios[i:i_end]
        # generate embeddings for all the audios in the batch
        _, emb = model.inference(batch)
        # create unique IDs
        ids = [f"{idx}" for idx in range(i, i_end)]
        # add all to upsert list
        to_upsert = list(zip(ids, emb.tolist()))
        # upsert/insert these records to pinecone
        _ = index.upsert(vectors=to_upsert)

    # check that we have all vectors in index
    index.describe_index_stats()


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


# Querying

Let's first listen to an audio from our dataset. We will generate embeddings for the audio and use it to find similar audios from the Pinecone index.

In [19]:
if RUN_AUDIO_DEMO:
    pass
    from IPython.display import Audio, display

    # we set an audio number to select from the dataset
    audio_num = 400
    # get the audio data of the audio number
    query_audio = data[audio_num]["audio"]["array"]
    # get the category of the audio number
    category = data[audio_num]["category"]
    # print the category and play the audio
    print("Query Audio:", category)
    Audio(query_audio, rate=32000)


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


In [20]:
if RUN_AUDIO_DEMO:
    pass
    # reshape query audio
    query_audio = query_audio[None, :]
    # get the embeddings for the audio from the model
    _, xq = model.inference(query_audio)
    xq.shape


We have now converted the audio into a 2048-dimension vector the same way we did for all the other audio we indexed. Let's use this to query our Pinecone index.

In [21]:
if RUN_AUDIO_DEMO:
    pass
    # query pinecone index with the query audio embeddings
    results = index.query(vector=xq[0].tolist(), top_k=3)
    results


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


In [22]:
if RUN_AUDIO_DEMO:
    pass
    # play the top 3 similar audios
    for r in results["matches"]:
        # select the audio data from the databse using the id as an index
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=32000))


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


In [23]:
if RUN_AUDIO_DEMO:
    pass
    def find_similar_audios(id):
        print("Query Audio:")
        # select the audio data from the databse using the id as an index
        query_audio = data[id]["audio"]["array"]
        # play the query audio
        display(Audio(query_audio, rate=32000))
        # query pinecone index with the query audio id
        result = index.query(id=str(id), top_k=5)
        print("Result:")
        # play the top 5 similar audios
        for r in result["matches"]:
            a = data[int(r["id"])]["audio"]["array"]
            display(Audio(a, rate=32000))


In [24]:
if RUN_AUDIO_DEMO:
    pass
    find_similar_audios(1642)


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


In [25]:
if RUN_AUDIO_DEMO:
    pass
    find_similar_audios(452)


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


In [26]:
if RUN_AUDIO_DEMO:
    pass
    #!wget https://storage.googleapis.com/audioset/miaow_16k.wav


We can load the audio into a Numpy array as follows:

In [27]:
if RUN_AUDIO_DEMO:
    pass
    import librosa

    a, _ = librosa.load("./data/miaow_16k.wav", sr=32000)
    Audio(a, rate=32000)


Now we generate the embeddings for this audio and query the Pinecone index.

In [28]:
if RUN_AUDIO_DEMO:
    pass
    # reshape query audio
    query_audio = a[None, :]
    # get the embeddings for the audio from the model
    _, xq = model.inference(query_audio)

    # query pinecone index with the query audio embeddings
    results = index.query(vector=xq[0].tolist(), top_k=3)

    # play the top 3 similar audios
    for r in results["matches"]:
        a = data[int(r["id"])]["audio"]["array"]
        display(Audio(a, rate=32000))


Résultat à observer après activation de la démonstration audio. Aucune inférence PANNs ni requête Pinecone n’a été exécutée pendant la validation locale.


# Delete the Index

Aucun effacement automatique du service distant. La démonstration conserve son index.


In [29]:
if RUN_AUDIO_DEMO:
    pass
    print('Index conservé. Aucun effacement automatique.')
